# Mini-ColBERTv2 on SciFact (Colab GPU)

Refactored pipeline: base → cad → evaluation.

- `run_base.py`: encode, index, 2-bit residual, IVF, search, output base candidates.
- `run_cad.py`: compute IDF + 4 ablations (Original/IDF/Adaptive/CAD).
- `run_evaluate.py`: print metrics table.

In [ ]:
# 1. Clone repo
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git
%cd colbert-hoangnd

In [ ]:
# 2. Install dependencies (pin transformers<5 to stay compatible with colbert-ai)
!pip install -q "transformers==4.57.6" "colbert-ai==0.2.22" scikit-learn datasets beir tqdm numpy ninja

In [ ]:
# 3. Ensure ninja is on PATH for colbert-ai C++ extension compilation
import os, sys
os.environ['PATH'] += ':' + os.path.join(sys.prefix, 'bin')
!ninja --version

In [ ]:
# 4. Run Base pipeline: checkpoint → index → search → base_candidates.pt
!python run_base.py --output_dir outputs --K 8192 --n_samples 200000 --ncells 4 --topk 100 --device cuda --doc_batch_size 64 --query_batch_size 64

In [ ]:
# 5. Verify base artifacts exist
import os
required = ['outputs/base_candidates.pt', 'outputs/rankings_original.json', 'outputs/query_embeddings.pt']
missing = [p for p in required if not os.path.exists(p)]
assert not missing, f'Missing files: {missing}'
print('Base pipeline complete.')

In [ ]:
# 6. Run CAD pipeline: only consumes base_candidates.pt + IDF
!python run_cad.py --output_dir outputs --topk 100

In [ ]:
# 7. Evaluate all variants
!python run_evaluate.py --output_dir outputs --qrels outputs/qrels_test.json